# Домашняя Работа 8

In [22]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib as mlp
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, random_split
import torch.nn as nn
import pandas as pd

In [20]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)  

Device: cuda


In [17]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

In [10]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [18]:
train_dataset_full = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    transform=train_transform,
    download=True
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    transform=test_transform,
    download=True
)

Files already downloaded and verified
Files already downloaded and verified


In [24]:
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

In [25]:
generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 40000
Val size: 10000
Test size: 10000


In [46]:
# Dataloader

BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers = 4,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    num_workers = 4,
    shuffle=False
)

In [47]:
# sanity check

x, y = next(iter(train_loader))

print("\nSanity check:")
print("Batch size:", x.shape[0])
print("x.shape:", x.shape)
print("y.shape:", y.shape)

print("Min value:", x.min().item())
print("Max value:", x.max().item())

print("Label example:", y[:10])


Sanity check:
Batch size: 256
x.shape: torch.Size([256, 3, 32, 32])
y.shape: torch.Size([256])
Min value: -1.0
Max value: 1.0
Label example: tensor([2, 7, 3, 9, 1, 1, 5, 5, 3, 0])


In [48]:
class MLP(nn.Module):
    def __init__(
        self,
        input_dim=3*32*32,
        hidden_dims=(256,),
        num_classes=10,
        dropout=0.0,
        batchnorm=False
    ):
        super().__init__()

        layers = []
        layers.append(nn.Flatten())

        in_dim = input_dim

        for h in hidden_dims:

            layers.append(nn.Linear(in_dim, h))

            if batchnorm:
                layers.append(nn.BatchNorm1d(h))

            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            in_dim = h

        layers.append(nn.Linear(in_dim, num_classes))

        self.model = nn.Sequential(*layers)


    def forward(self, x):
        return self.model(x)

In [49]:
def accuracy(logits, y):

    preds = torch.argmax(logits, dim=1)

    correct = (preds == y).sum().item()

    return correct / y.size(0)

In [50]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0
    total_acc = 0
    total_samples = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        batch_size = x.size(0)

        total_loss += loss.item() * batch_size
        total_acc += accuracy(logits, y) * batch_size
        total_samples += batch_size


    return (
        total_loss / total_samples,
        total_acc / total_samples
    )

In [51]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    total_acc = 0
    total_samples = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        loss = criterion(logits, y)

        batch_size = x.size(0)

        total_loss += loss.item() * batch_size
        total_acc += accuracy(logits, y) * batch_size
        total_samples += batch_size


    return (
        total_loss / total_samples,
        total_acc / total_samples
    )

In [53]:
def train_model(
    model,
    train_loader,
    val_loader,
    epochs,
    lr,
    device
):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_acc = 0
    best_epoch = 0
    best_state = None


    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            criterion,
            device
        )


        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)


        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train loss {train_loss:.4f} acc {train_acc:.4f}")
        print(f"Val   loss {val_loss:.4f} acc {val_acc:.4f}")
        print()


        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_epoch = epoch

            best_state = model.state_dict()


    return history, best_state, best_val_acc, best_epoch

In [54]:
# Early Stopping

def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    lr,
    device,
    patience=5,
    max_epochs=50
):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_acc = 0
    best_state = None

    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }


    for epoch in range(max_epochs):

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )


        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)


        if val_acc > best_acc:

            best_acc = val_acc
            best_state = model.state_dict()

            patience_counter = 0

        else:

            patience_counter += 1


        if patience_counter >= patience:

            print("Early stopping triggered")
            break


    return history, best_state, best_acc

In [55]:
experiments = {

    "E0": dict(
        hidden_dims=(256,),
        dropout=0.0,
        batchnorm=False
    ),

    "E1": dict(
        hidden_dims=(512, 256),
        dropout=0.0,
        batchnorm=False
    ),

    "E2": dict(
        hidden_dims=(512, 256),
        dropout=0.3,
        batchnorm=False
    ),

    "E3": dict(
        hidden_dims=(512, 256),
        dropout=0.0,
        batchnorm=True
    ),
}

In [56]:
results = {}

for exp_id, cfg in experiments.items():

    print("Running", exp_id)

    model = MLP(
        hidden_dims=cfg["hidden_dims"],
        dropout=cfg["dropout"],
        batchnorm=cfg["batchnorm"]
    ).to(device)


    history, best_state, best_acc, best_epoch = train_model(
        model,
        train_loader,
        val_loader,
        epochs=20,
        lr=1e-3,
        device=device
    )


    results[exp_id] = {
        "best_acc": best_acc,
        "best_state": best_state,
        "config": cfg,
        "history": history
    }

Running E0
Epoch 1/20
Train loss 1.6756 acc 0.4089
Val   loss 1.5499 acc 0.4560

Epoch 2/20
Train loss 1.4662 acc 0.4884
Val   loss 1.4673 acc 0.4870

Epoch 3/20
Train loss 1.3745 acc 0.5219
Val   loss 1.4475 acc 0.4986

Epoch 4/20
Train loss 1.2941 acc 0.5470
Val   loss 1.4416 acc 0.5027

Epoch 5/20
Train loss 1.2329 acc 0.5723
Val   loss 1.4092 acc 0.5198

Epoch 6/20
Train loss 1.1783 acc 0.5933
Val   loss 1.4365 acc 0.5110

Epoch 7/20
Train loss 1.1237 acc 0.6117
Val   loss 1.3957 acc 0.5265

Epoch 8/20
Train loss 1.0727 acc 0.6300
Val   loss 1.4102 acc 0.5245

Epoch 9/20
Train loss 1.0315 acc 0.6442
Val   loss 1.4443 acc 0.5225

Epoch 10/20
Train loss 0.9915 acc 0.6586
Val   loss 1.4717 acc 0.5255

Epoch 11/20
Train loss 0.9334 acc 0.6784
Val   loss 1.4662 acc 0.5264

Epoch 12/20
Train loss 0.9011 acc 0.6915
Val   loss 1.4748 acc 0.5247

Epoch 13/20
Train loss 0.8545 acc 0.7087
Val   loss 1.5010 acc 0.5216

Epoch 14/20
Train loss 0.8230 acc 0.7196
Val   loss 1.5281 acc 0.5202

Epoc

In [57]:
best_exp = max(results, key=lambda x: results[x]["best_acc"])

print("Best experiment:", best_exp)

Best experiment: E2


In [58]:
cfg = results[best_exp]["config"]

model = MLP(
    hidden_dims=cfg["hidden_dims"],
    dropout=cfg["dropout"],
    batchnorm=cfg["batchnorm"]
).to(device)


history, best_state, best_acc = train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    lr=1e-3,
    device=device,
    patience=5
)

Early stopping triggered


In [79]:
import os

ARTIFACTS_DIR = "artifacts"
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [80]:
import csv

def save_runs_csv(results, filepath, dataset="CIFAR10", seed=42):

    fieldnames = [
        "experiment_id",
        "dataset",
        "seed",
        "model_summary",
        "dropout",
        "batchnorm",
        "epochs_trained",
        "best_val_accuracy",
        "best_val_loss"
    ]

    with open(filepath, mode="w", newline="") as f:

        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for exp_id, res in results.items():

            cfg = res["config"]
            history = res["history"]

            best_epoch = max(
                range(len(history["val_acc"])),
                key=lambda i: history["val_acc"][i]
            )

            writer.writerow({
                "experiment_id": exp_id,
                "dataset": dataset,
                "seed": seed,
                "model_summary": str(cfg["hidden_dims"]),
                "dropout": cfg["dropout"],
                "batchnorm": cfg["batchnorm"],
                "epochs_trained": len(history["train_loss"]),
                "best_val_accuracy": history["val_acc"][best_epoch],
                "best_val_loss": history["val_loss"][best_epoch]
            })

In [81]:
save_runs_csv(
    results,
    os.path.join(ARTIFACTS_DIR, "runs.csv")
)

In [82]:
def save_best_model(best_state, filepath):

    torch.save(best_state, filepath)

save_best_model(
    best_state,
    os.path.join(ARTIFACTS_DIR, "best_model.pt")
)

In [83]:
best_state = results[best_exp]["best_state"]
best_config = results[best_exp]["config"]
best_history = results[best_exp]["history"]

In [84]:
import json

def save_best_config(config, filepath, dataset="CIFAR10", seed=42):

    full_config = {
        "dataset": dataset,
        "seed": seed,
        "model": {
            "hidden_dims": config["hidden_dims"],
            "dropout": config["dropout"],
            "batchnorm": config["batchnorm"]
        },
        "optimizer": "Adam",
        "loss": "CrossEntropyLoss"
    }

    with open(filepath, "w") as f:

        json.dump(full_config, f, indent=4)

In [85]:
save_best_config(
    best_config,
    os.path.join(ARTIFACTS_DIR, "best_config.json")
)

In [86]:
import matplotlib.pyplot as plt

def save_curves(history, filepath):

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.plot(epochs, history["train_loss"], label="train")
    plt.plot(epochs, history["val_loss"], label="val")
    plt.title("Loss")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, history["train_acc"], label="train")
    plt.plot(epochs, history["val_acc"], label="val")
    plt.title("Accuracy")
    plt.legend()

    plt.savefig(filepath)
    plt.close()

In [87]:
save_curves(
    best_history,
    os.path.join(FIGURES_DIR, "curves_best.png")
)